In [ ]:
%cd ..

In [ ]:
from pathlib import Path
from molrgen.data.pydantic_dataset import read_jsonl
import json
from tqdm.auto import tqdm

dataset = read_jsonl(Path("data/property_prediction/eval_prompts_boxed.jsonl"))
id_to_metadata = {
    line.identifier: line.conversations[0].meta for line in dataset
}

In [ ]:
batch_api_results_path = Path("batch_api_results")

def process_line_iterator(line):
    id = line["custom_id"]
    if "||" in id:
        id = custom_id.split("||")[0]
    for choice in line["response"]["body"]["choices"]:
        messages_content = choice["message"]["content"]
        completion = ""
        for message in messages_content:
            if "thinking" in message:
                for think_content in message["thinking"]:
                    completion += "\n" + think_content["text"]
            elif "text" in message:
                completion += "\n" + message["text"]
            else:
                raise ValueError("Message content must have either 'thinking' or 'text' key")
        yield dict(
            input = None,
            output = completion,
            metadata = id_to_metadata[id],
        )

all_results = []
for file in tqdm(batch_api_results_path.iterdir(), total=len(list(batch_api_results_path.iterdir()))):
    #Load jsonl file
    with open(file, "r") as f:
        for raw_line in f:
            line = json.loads(raw_line)
            for completion in process_line_iterator(line):
                all_results.append(completion)


In [ ]:
len(all_results)

In [ ]:
# save to jsonl file
output_path = Path("MolGenOutput/property_prediction/Mistral-Small-4-128B/out_0.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    for result in all_results:
        f.write(json.dumps(result) + "\n")